In [129]:
import pandas as pd

# -----------------------------
# Load GFF3 file
# -----------------------------
df = pd.read_csv(
    "Homo_sapiens.GRCh37.87.chr.gff3.gz",
    sep="\t",
    compression="gzip",
    comment="#",
    header=None
)
df.columns = [
    "seqid", "source", "type", "start", "end",
    "score", "strand", "phase", "attributes"
]

# Make sure numeric
df["seqid"] = df["seqid"].astype(str)
df["start"] = df["start"].astype(int)
df["end"] = df["end"].astype(int)

# -----------------------------
# Load variant file
# -----------------------------
variants_df = pd.read_csv("case_variants_gene_orpha_inher.csv") 
def get_causal_cds(gff_df, chrom, position):
    # Filter by chromosome
    chr_df = gff_df[gff_df["seqid"] == str(chrom)]
    
    # Keep only CDS
    cds_df = chr_df[chr_df["type"] == "CDS"].copy()
    
    # ---------------------
    # 1️⃣ Primary: CDS overlapping the variant
    primary = cds_df[
        (cds_df["start"] <= position) &
        (cds_df["end"] >= position)
    ].copy()
    
    if not primary.empty:
        cds_df = primary
    else:
        # ---------------------
        # 2️⃣ Fallback: start <= position+10 & end >= position
        fallback1 = cds_df[
            (cds_df["start"] <= position + 10) &
            (cds_df["end"] >= position)
        ].copy()
        if not fallback1.empty:
            cds_df = fallback1
        else:
            # ---------------------
            # 3️⃣ Fallback: end <= position+10
            fallback2 = cds_df[
                (cds_df["end"] <= position + 10)
            ].copy()
            if not fallback2.empty:
                # Keep only the row with largest 'end'
                max_end_idx = fallback2["end"].idxmax()
                cds_df = fallback2.loc[[max_end_idx]]
            else:
                # ---------------------
                # No CDS found, return empty fields
                return pd.DataFrame([{
                    "CHROM": chrom,
                    "POS": position,
                    "START": "",
                    "END": "",
                    "STRAND": ""
                }])
    
    # ---------------------
    # Compute length and select largest CDS (for primary/fallback1)
    if len(cds_df) > 1:
        cds_df["length"] = cds_df["end"] - cds_df["start"]
        largest_cds = cds_df.loc[cds_df["length"].idxmax()]
    else:
        largest_cds = cds_df.iloc[0]
    
    # Return as dataframe
    return pd.DataFrame([{
        "CHROM": chrom,
        "POS": position,
        "START": largest_cds["start"],
        "END": largest_cds["end"],
        "STRAND": largest_cds["strand"]
    }])

output_list = []

for idx, row in variants_df.iterrows():
    chrom = row["CHROM"]
    pos = row["POS"]
    output_list.append(get_causal_cds(df, chrom, pos))

# Concatenate all results
final_output_df = pd.concat(output_list, ignore_index=True)

# Save to CSV
final_output_df.to_csv("causal_variant_output.csv", index=False)

print(final_output_df.head())

/var/folders/fq/f9fql1ns42j64klkb0rn71l80000gn/T/ipykernel_26494/1012224879.py:6: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


   CHROM        POS      START        END STRAND
0      4    1807371    1807286    1807396      +
1      7  138391446  138391369  138391462      -
2      1  154574727  154573517  154575102      -
3      2  163128866  163128736  163128897      -
4     16    1505761    1505732    1505796      -


In [7]:
import pandas as pd
df_seq = pd.read_csv("chrom_pos_seq.csv")
print(df_seq.head())
print(len(df_seq))

   CHROM        POS  POS_minus_10  POS_plus_10                    SEQ
0   chr4    1807371       1807361      1807381  ACATCATCAACCTGCTGGGCG
1   chr7  138391446     138391436    138391456  ACATAGAACTTGTTCTGGAAC
2   chr1  154574727     154574717    154574737  CTCAGTTCCTGGAAATGTGAG
3   chr2  163128866     163128856    163128876  CAGGACGTAGGTGCTCTCATC
4  chr16    1505761       1505751      1505771  TGGTTCCAGAAGGACGCACCC
1508


In [9]:
df_pos = pd.read_csv("causal_variant_start_end.csv")
print(df_pos.head())
print(len(df_pos))

   CHROM        POS      START        END STRAND
0      4    1807371    1807286    1807396      +
1      7  138391446  138391369  138391462      -
2      1  154574727  154573517  154575102      -
3      2  163128866  163128736  163128897      -
4     16    1505761    1505732    1505796      -
1508


In [69]:
df_case = pd.read_csv("case_variants_gene_orpha_inher.csv")
df_case

,CHROM,POS,ID,REF,ALT,GENEINFO,ORPHACODE,INHERITANCE
0,4,1807371,16337,C,A,FGFR3,15,AD
1,7,138391446,1344704,T,C,ATP6V0A4,18,AD
2,1,154574727,2942810,G,A,ADAR,41,AD
3,2,163128866,812537,G,C,IFIH1,51,AD
4,16,1505761,1012215,A,G,CLCN7,53,AD
...,...,...,...,...,...,...,...,...
1503,15,52425575,2575918,C,T,GNB5,542306,AR
1504,17,8076848,929285,A,C,TMEM107,542310,AR
1505,1,150990382,932732,T,C,PRUNE1,544469,AR
1506,11,17167409,599543,C,A,PIK3C2A,557003,AR


In [31]:
# -----------------------------
df_seq["CHROM"] = df_seq["CHROM"].str.replace("chr", "", regex=False)

# Ensure both are string
df_seq["CHROM"] = df_seq["CHROM"].astype(str)
df_pos["CHROM"] = df_pos["CHROM"].astype(str)

df_seq["POS"] = df_seq["POS"].astype(int)
df_pos["POS"] = df_pos["POS"].astype(int)

In [33]:
df_merged = pd.concat(
    [df_seq.reset_index(drop=True),
     df_pos.drop(columns=["CHROM", "POS"]).reset_index(drop=True)],
    axis=1
)

print(len(df_merged))

1508


In [35]:
df_merged.to_csv("case_variants_new_mutation.csv", index=False)

In [41]:
import pandas as pd

# -------------------------
# Load CSV
# -------------------------
df = pd.read_csv("case_variants_new_mutation.csv")

# -------------------------
# Helper functions
# -------------------------
def reverse_complement(seq):
    complement = str.maketrans("ACGT", "TGCA")
    return seq.translate(complement)[::-1]

def translate_codon(codon):
    codon_table = {
        'ATA':'I','ATC':'I','ATT':'I','ATG':'M',
        'ACA':'T','ACC':'T','ACG':'T','ACT':'T',
        'AAC':'N','AAT':'N','AAA':'K','AAG':'K',
        'AGC':'S','AGT':'S','AGA':'R','AGG':'R',
        'CTA':'L','CTC':'L','CTG':'L','CTT':'L',
        'CCA':'P','CCC':'P','CCG':'P','CCT':'P',
        'CAC':'H','CAT':'H','CAA':'Q','CAG':'Q',
        'CGA':'R','CGC':'R','CGG':'R','CGT':'R',
        'GTA':'V','GTC':'V','GTG':'V','GTT':'V',
        'GCA':'A','GCC':'A','GCG':'A','GCT':'A',
        'GAC':'D','GAT':'D','GAA':'E','GAG':'E',
        'GGA':'G','GGC':'G','GGG':'G','GGT':'G',
        'TCA':'S','TCC':'S','TCG':'S','TCT':'S',
        'TTC':'F','TTT':'F','TTA':'L','TTG':'L',
        'TAC':'Y','TAT':'Y','TAA':'*','TAG':'*',
        'TGC':'C','TGT':'C','TGA':'*','TGG':'W'
    }
    return codon_table.get(codon.upper(), 'X')

def find_codon_and_aa(seq, index, frame_offset):
    codon_start = ((index + frame_offset) // 3) * 3 - frame_offset
    codon = seq[codon_start:codon_start+3]
    if len(codon) < 3:
        codon += 'N' * (3 - len(codon))
    aa = translate_codon(codon)
    return codon, aa

def generate_three_mutations(seq, variant_index, frame_offset):
    bases = ['A','C','G','T']
    mutations = []

    for i, base in enumerate(seq):
        if i == variant_index:
            continue

        original_codon, original_aa = find_codon_and_aa(seq, i, frame_offset)

        for b in bases:
            if b != base:
                new_seq = seq[:i] + b + seq[i+1:]
                _, new_aa = find_codon_and_aa(new_seq, i, frame_offset)

                if new_aa != original_aa:
                    mutations.append((new_seq, i, base, b))
                    break  # only one change per position

        if len(mutations) == 3:
            break

    return mutations

# -------------------------
# Process each row
# -------------------------
results = []

for _, row in df.iterrows():
    seq = row['SEQ']
    strand = row['STRAND']
    pos = row['POS']
    start = row['START']
    end = row['END']
    pos_minus_10 = row['POS_minus_10']
    pos_plus_10 = row['POS_plus_10']

    if strand == "+":
        seq_in_frame = seq
        frame_offset = pos_minus_10 - start
    else:
        seq_in_frame = reverse_complement(seq)
        frame_offset = end - pos_plus_10

    variant_index = pos - pos_minus_10

    mutations = generate_three_mutations(seq_in_frame, variant_index, frame_offset)

    for new_seq, changed_index, original_allele, changed_allele in mutations:

        original_codon, original_aa = find_codon_and_aa(seq_in_frame, changed_index, frame_offset)
        changed_codon, changed_aa = find_codon_and_aa(new_seq, changed_index, frame_offset)

        pos_of_changed_allele = (
            pos_minus_10 + changed_index
            if strand == "+"
            else pos_plus_10 - changed_index
        )

        results.append({
            'CHROM': row['CHROM'],      # repeated 3 times
            'POS': pos,                 # repeated 3 times
            'POS_changed_allele': pos_of_changed_allele,
            'Original_allele': original_allele,
            'Changed_allele': changed_allele,
            'Original_codon': original_codon,
            'Changed_codon': changed_codon,
            'Original_AA': original_aa,
            'Changed_AA': changed_aa
        })

# -------------------------
# Save results
# -------------------------
df_results = pd.DataFrame(results)
df_results.to_csv("case_variants_new_mutation_3.csv", index=False)

print("Finished. Output rows:", len(df_results))

Finished. Output rows: 4524


In [85]:
df_mutation3 = pd.read_csv("case_variants_new_mutation_3.csv")
df_mutation3

,CHROM,POS,POS_changed_allele,Original_allele,Changed_allele,Original_codon,Changed_codon,Original_AA,Changed_AA
0,4,1807371,1807361,A,C,ACA,CCA,T,P
1,4,1807371,1807362,C,A,ACA,AAA,T,K
2,4,1807371,1807364,T,A,TCA,ACA,S,T
3,7,138391446,138391456,G,A,GTT,ATT,V,I
4,7,138391446,138391455,T,A,GTT,GAT,V,D
...,...,...,...,...,...,...,...,...,...
4519,11,17167409,17167416,T,A,ATG,AAG,M,K
4520,11,17167409,17167415,G,A,ATG,ATA,M,I
4521,5,74055190,74055200,T,A,TAG,AAG,*,K
4522,5,74055190,74055199,A,C,TAG,TCG,*,S


In [87]:
import pandas as pd

# Repeat each row of df_case 3 times
df_case_repeated = df_case.loc[df_case.index.repeat(3)].reset_index(drop=True)

# Reset index of df_merged to align properly
df_mutation3 = df_mutation3.reset_index(drop=True)

# Copy columns into df_merged
df_mutation3[["GENEINFO", "ORPHACODE", "INHERITANCE"]] = \
    df_case_repeated[["GENEINFO", "ORPHACODE", "INHERITANCE"]]

df_mutation3

,CHROM,POS,POS_changed_allele,Original_allele,Changed_allele,Original_codon,Changed_codon,Original_AA,Changed_AA,GENEINFO,ORPHACODE,INHERITANCE
0,4,1807371,1807361,A,C,ACA,CCA,T,P,FGFR3,15,AD
1,4,1807371,1807362,C,A,ACA,AAA,T,K,FGFR3,15,AD
2,4,1807371,1807364,T,A,TCA,ACA,S,T,FGFR3,15,AD
3,7,138391446,138391456,G,A,GTT,ATT,V,I,ATP6V0A4,18,AD
4,7,138391446,138391455,T,A,GTT,GAT,V,D,ATP6V0A4,18,AD
...,...,...,...,...,...,...,...,...,...,...,...,...
4519,11,17167409,17167416,T,A,ATG,AAG,M,K,PIK3C2A,557003,AR
4520,11,17167409,17167415,G,A,ATG,ATA,M,I,PIK3C2A,557003,AR
4521,5,74055190,74055200,T,A,TAG,AAG,*,K,GFM2,565624,AR
4522,5,74055190,74055199,A,C,TAG,TCG,*,S,GFM2,565624,AR


In [89]:
# Drop unwanted columns
df_mutation3 = df_mutation3.drop(columns=[
    "POS",
    "Original_codon",
    "Changed_codon",
    "Original_AA",
    "Changed_AA"
])

# Rename columns
df_mutation3 = df_mutation3.rename(columns={
    "POS_changed_allele": "POS",
    "Original_allele": "REF",
    "Changed_allele": "ALT"
})

# Insert ID column between POS and REF
df_mutation3.insert(
    df_mutation3.columns.get_loc("REF"),
    "ID",
    "."
)

In [91]:
print(df_mutation3)

      CHROM        POS ID REF ALT  GENEINFO  ORPHACODE INHERITANCE
0         4    1807361  .   A   C     FGFR3         15          AD
1         4    1807362  .   C   A     FGFR3         15          AD
2         4    1807364  .   T   A     FGFR3         15          AD
3         7  138391456  .   G   A  ATP6V0A4         18          AD
4         7  138391455  .   T   A  ATP6V0A4         18          AD
...     ...        ... ..  ..  ..       ...        ...         ...
4519     11   17167416  .   T   A   PIK3C2A     557003          AR
4520     11   17167415  .   G   A   PIK3C2A     557003          AR
4521      5   74055200  .   T   A      GFM2     565624          AR
4522      5   74055199  .   A   C      GFM2     565624          AR
4523      5   74055198  .   G   C      GFM2     565624          AR

[4524 rows x 8 columns]


In [93]:
# Updating multiple columns in one assignment operation to avoid SettingWithCopyWarning
df_mutation3 = df_mutation3.assign(
    QUAL='.',
    FILTER='.',
    INFO='PR',
    FORMAT='GT'
)
df_mutation3

,CHROM,POS,ID,REF,ALT,GENEINFO,ORPHACODE,INHERITANCE,QUAL,FILTER,INFO,FORMAT
0,4,1807361,.,A,C,FGFR3,15,AD,.,.,PR,GT
1,4,1807362,.,C,A,FGFR3,15,AD,.,.,PR,GT
2,4,1807364,.,T,A,FGFR3,15,AD,.,.,PR,GT
3,7,138391456,.,G,A,ATP6V0A4,18,AD,.,.,PR,GT
4,7,138391455,.,T,A,ATP6V0A4,18,AD,.,.,PR,GT
...,...,...,...,...,...,...,...,...,...,...,...,...
4519,11,17167416,.,T,A,PIK3C2A,557003,AR,.,.,PR,GT
4520,11,17167415,.,G,A,PIK3C2A,557003,AR,.,.,PR,GT
4521,5,74055200,.,T,A,GFM2,565624,AR,.,.,PR,GT
4522,5,74055199,.,A,C,GFM2,565624,AR,.,.,PR,GT


In [97]:
import pandas as pd
import numpy as np

# Number of variants (groups of 3 rows)
num_variants = len(df_mutation3) // 3

# 1️⃣ Create 1508 genotype columns initialized to "0/0"
genotype_cols = pd.DataFrame(
    "0/0",
    index=df_mutation3.index,
    columns=[f'col_{i+1}' for i in range(num_variants)]
)

df_mutation3 = pd.concat([df_mutation3, genotype_cols], axis=1)

# 2️⃣ Assign genotype per variant block
for i in range(num_variants):

    # Determine row range for this variant
    row_start = i * 3
    row_end = row_start + 3

    # Inheritance type (same for the 3 rows)
    inheritance = df_mutation3.loc[row_start, "INHERITANCE"]

    # Determine genotype
    if inheritance == "AD":
        genotype = np.random.choice(["0/1", "1/0"])
    elif inheritance == "AR":
        genotype = "1/1"
    else:
        genotype = "0/0"

    # Column name
    col_name = f'col_{i+1}'

    # Assign genotype to the 3 rows in that column
    df_mutation3.loc[row_start:row_end-1, col_name] = genotype

# Optional: defragment memory
df_mutation3 = df_mutation3.copy()

df_mutation3

,CHROM,POS,ID,REF,ALT,GENEINFO,ORPHACODE,INHERITANCE,QUAL,FILTER,...,col_1499,col_1500,col_1501,col_1502,col_1503,col_1504,col_1505,col_1506,col_1507,col_1508
0,4,1807361,.,A,C,FGFR3,15,AD,.,.,...,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0
1,4,1807362,.,C,A,FGFR3,15,AD,.,.,...,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0
2,4,1807364,.,T,A,FGFR3,15,AD,.,.,...,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0
3,7,138391456,.,G,A,ATP6V0A4,18,AD,.,.,...,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0
4,7,138391455,.,T,A,ATP6V0A4,18,AD,.,.,...,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4519,11,17167416,.,T,A,PIK3C2A,557003,AR,.,.,...,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0,1/1,0/0
4520,11,17167415,.,G,A,PIK3C2A,557003,AR,.,.,...,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0,1/1,0/0
4521,5,74055200,.,T,A,GFM2,565624,AR,.,.,...,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0,1/1
4522,5,74055199,.,A,C,GFM2,565624,AR,.,.,...,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0,1/1


In [100]:
# Removing the specified columns "GENEINFO", "ORPHACODE", and "INHERITANCE"
df_mutation3 = df_mutation3.drop(columns=["GENEINFO", "ORPHACODE", "INHERITANCE"])
df_mutation3

,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,FORMAT,col_1,...,col_1499,col_1500,col_1501,col_1502,col_1503,col_1504,col_1505,col_1506,col_1507,col_1508
0,4,1807361,.,A,C,.,.,PR,GT,1/0,...,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0
1,4,1807362,.,C,A,.,.,PR,GT,1/0,...,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0
2,4,1807364,.,T,A,.,.,PR,GT,1/0,...,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0
3,7,138391456,.,G,A,.,.,PR,GT,0/0,...,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0
4,7,138391455,.,T,A,.,.,PR,GT,0/0,...,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4519,11,17167416,.,T,A,.,.,PR,GT,0/0,...,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0,1/1,0/0
4520,11,17167415,.,G,A,.,.,PR,GT,0/0,...,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0,1/1,0/0
4521,5,74055200,.,T,A,.,.,PR,GT,0/0,...,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0,1/1
4522,5,74055199,.,A,C,.,.,PR,GT,0/0,...,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0,0/0,1/1


In [103]:
df_mutation3.to_csv('variants_new_vcf.csv', index=False)